# Монгол F5-TTS fine-tune (Google Colab)

**Өмнө нь:** `mn_f5.zip`-ийг Google Drive-ийнхаа үндсэн (MyDrive) фолдерт байршуул.

**Colab тохиргоо:** дээд талаас `Runtime → Change runtime type → GPU` (T4) сонго.

Дараа нь нүд бүрийг дарааллан ▶ дарж ажиллуул. Алдаа гарвал зогсоо, надад хэлээрэй — хамт засна.

## 1. GPU шалгах

In [ ]:
!nvidia-smi

## 2. F5-TTS суулгах (~3-5 мин)

In [ ]:
!git clone https://github.com/SWivid/F5-TTS
%cd F5-TTS
!pip install -q -e .

## 3. Google Drive холбож, dataset задлах

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# mn_f5.zip-ийг MyDrive-д тавьсан гэж үзэв. Өөр газар бол замаа зас.
!unzip -q -o /content/drive/MyDrive/mn_f5.zip -d /content/mn_f5
!echo '--- агуулга ---'; ls /content/mn_f5
!echo '--- metadata эхний мөр ---'; head -2 /content/mn_f5/metadata.csv
!echo '--- wav тоо ---'; ls /content/mn_f5/wavs | wc -l

## 4. Dataset-ийг F5 формат руу бэлтгэх

`metadata.csv` (audio|text) → F5-ийн arrow dataset болгоно.

In [ ]:
!python src/f5_tts/train/datasets/prepare_csv_wavs.py /content/mn_f5 /content/F5-TTS/data/mn_char
!ls /content/F5-TTS/data/mn_char

## 5. Fine-tune эхлүүлэх (Gradio UI — хамгийн хялбар)

Доорх нүдийг ажиллуулбал **public холбоос** (`.gradio.live`) гарна. Тэр холбоосыг нээж:
1. **Train** таб → dataset-аа сонго (`mn_char`)
2. Base model: **F5TTS_v1_Base** (fine-tune)
3. Epochs/learning rate-ийг анхдагчаар үлдээж эхэл
4. **Start Training** дар → loss буурч байгааг хар

GPU-гийн санах ой бага бол `Batch size`-ийг багасга.

In [ ]:
!f5-tts_finetune-gradio --share

## 6. (Хувилбар) CLI-аар fine-tune

Gradio ажиллахгүй бол энэ CLI-г оролдоно (аргументуудыг F5 хувилбараас хамаарч тааруулж магадгүй):

In [ ]:
!accelerate launch src/f5_tts/train/finetune_cli.py \
  --exp_name F5TTS_v1_Base \
  --dataset_name mn_char \
  --finetune \
  --learning_rate 1e-5 \
  --batch_size_per_gpu 2000 \
  --epochs 40 \
  --save_per_updates 2000 \
  --last_per_updates 1000

## 7. Сургалт дууссаны дараа

Checkpoint-ууд `ckpts/mn_char/` дотор үүснэ. Тэдгээрийг Drive рүү хуулж хадгал:
```python
!cp -r ckpts/mn_char /content/drive/MyDrive/mn_f5_ckpts
```
Дараа нь энэ загвараар Монгол текстийг оригинал хоолойгоор уншуулж туршина (inference).